# Swiggy Restaurant Data Cleaning & KPI Analysis

**Objective**: Clean the raw Swiggy dataset and identify key performance indicators (KPIs) for a future Power BI dashboard.

**Dataset**: Restaurant listings from Bangalore areas (Koramangala, HSR, BTM etc.) with Shop Name, Cuisine, Location, Rating, Cost for Two.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)
sns.set_style('whitegrid')
%matplotlib inline


In [ ]:
# Load the data
df = pd.read_excel('/workspace/attachments/swiggy.xls')
print("Original Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nNull Values:")
print(df.isnull().sum())


In [ ]:
print("Unique Ratings:", df['Rating'].unique())
print("\nCost for Two samples:", df['Cost_for_Two'].unique()[:10])
print("\nLocation value counts (top 15):")
print(df['Location'].value_counts().head(15))
print("\nDuplicate rows:", df.duplicated().sum())


## Data Cleaning

### Issues Identified:
1. **Cost_for_Two**: Contains garbled rupee symbol (`â‚¹`) instead of `₹`
2. **Rating**: Contains `'--'` for missing values; stored as object
3. **Location**: Inconsistent formatting (e.g., "Koramangala, Koramangala", "5th Block, Koramangala")
4. **Cuisine**: Multi-valued (comma-separated)
5. All columns are object type


In [ ]:
# 1. Clean Cost_for_Two
import re

def clean_cost(cost_str):
    if pd.isna(cost_str):
        return np.nan
    # Extract only ASCII digits (avoids garbled rupee symbol issues)
    cleaned = re.sub(r'[^0-9]', '', str(cost_str))
    return int(cleaned) if cleaned else np.nan

df['Cost_for_Two'] = df['Cost_for_Two'].apply(clean_cost)
print("Cost_for_Two after cleaning:")
print(df['Cost_for_Two'].describe())
print("\nSample cleaned costs:", df['Cost_for_Two'].head(10).tolist())


In [ ]:
# 2. Clean Rating
df['Rating'] = df['Rating'].replace('--', np.nan)
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
print("Rating after cleaning:")
print(df['Rating'].describe())
print("\nMissing ratings:", df['Rating'].isnull().sum())


In [ ]:
# 3. Extract standardized Area from Location
def extract_area(location):
    loc = str(location).lower()
    if 'koramangala' in loc or 'kormangala' in loc:
        return 'Koramangala'
    elif 'hsr' in loc:
        return 'HSR'
    elif 'btm' in loc or 'btm' in loc:
        return 'BTM'
    elif 'jayanagar' in loc:
        return 'Jayanagar'
    else:
        return 'Other'

df['Area'] = df['Location'].apply(extract_area)
print("Restaurants by Area:")
print(df['Area'].value_counts())


In [ ]:
# 4. Clean Shop_Name
df['Shop_Name'] = df['Shop_Name'].str.strip().str.title()
print("Sample cleaned shop names:")
print(df['Shop_Name'].head(10).tolist())


In [ ]:
# 5. Process Cuisine (for analysis)
# Create a cleaned cuisine string
df['Cuisine_Clean'] = df['Cuisine'].str.replace(r'\s*,\s*', ', ', regex=True).str.strip()

# For multi-hot or top cuisines later
print("Sample cuisines:")
print(df['Cuisine_Clean'].head(5).tolist())


In [ ]:
# 6. Drop exact duplicates and reset index
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

# Final cleaned columns
cleaned_df = df[['Shop_Name', 'Cuisine_Clean', 'Location', 'Area', 'Rating', 'Cost_for_Two']].copy()
cleaned_df = cleaned_df.rename(columns={'Cuisine_Clean': 'Cuisine'})
print("\nCleaned Data Preview:")
display(cleaned_df.head(10))
print("\nData Types after cleaning:")
print(cleaned_df.dtypes)


## Key Performance Indicators (KPIs)

Based on the cleaned dataset, the following KPIs are useful for a restaurant analytics dashboard:


In [ ]:
# KPI 1: Overall Summary
print("=== OVERALL KPIs ===")
print(f"Total Restaurants: {len(cleaned_df)}")
print(f"Average Rating: {cleaned_df['Rating'].mean():.2f}")
print(f"Median Rating: {cleaned_df['Rating'].median():.2f}")
print(f"Average Cost for Two: ₹{cleaned_df['Cost_for_Two'].mean():.0f}")
print(f"Median Cost for Two: ₹{cleaned_df['Cost_for_Two'].median():.0f}")
print(f"Highest Rated: {cleaned_df['Rating'].max()}")
print(f"Lowest Rated (non-null): {cleaned_df['Rating'].min()}")


In [ ]:
# KPI 2: Area-wise Performance
area_kpi = cleaned_df.groupby('Area').agg(
    Restaurant_Count=('Shop_Name', 'count'),
    Avg_Rating=('Rating', 'mean'),
    Avg_Cost=('Cost_for_Two', 'mean'),
    Median_Cost=('Cost_for_Two', 'median')
).round(2).sort_values('Restaurant_Count', ascending=False)

print("=== AREA-WISE KPIs ===")
display(area_kpi)


In [ ]:
# KPI 3: Top 10 Highest Rated Restaurants
top_rated = cleaned_df.nlargest(10, 'Rating')[['Shop_Name', 'Area', 'Cuisine', 'Rating', 'Cost_for_Two']]
print("=== TOP 10 HIGHEST RATED ===")
display(top_rated)


In [ ]:
# KPI 4: Best Value Restaurants (High Rating, Low Cost)
# Rating >= 4.2 and Cost <= 300
best_value = cleaned_df[
    (cleaned_df['Rating'] >= 4.2) & 
    (cleaned_df['Cost_for_Two'] <= 300)
].sort_values(['Rating', 'Cost_for_Two'], ascending=[False, True])

print(f"=== BEST VALUE RESTAURANTS (Rating ≥ 4.2 & Cost ≤ ₹300): {len(best_value)} ===")
display(best_value[['Shop_Name', 'Area', 'Rating', 'Cost_for_Two']].head(15))


In [ ]:
# KPI 5: Top Cuisines (by frequency)
# Split cuisines and count
all_cuisines = []
for cuisines in cleaned_df['Cuisine'].dropna():
    for c in cuisines.split(', '):
        all_cuisines.append(c.strip())

cuisine_counts = Counter(all_cuisines)
top_cuisines = pd.DataFrame(cuisine_counts.most_common(15), columns=['Cuisine', 'Count'])
print("=== TOP 15 CUISINES ===")
display(top_cuisines)


In [ ]:
# KPI 6: Rating Distribution
rating_dist = cleaned_df['Rating'].value_counts().sort_index()
print("=== RATING DISTRIBUTION ===")
print(rating_dist)
print(f"\nRestaurants with Rating ≥ 4.0: {(cleaned_df['Rating'] >= 4.0).sum()} ({(cleaned_df['Rating'] >= 4.0).mean()*100:.1f}%)")
print(f"Restaurants with Rating ≥ 4.3: {(cleaned_df['Rating'] >= 4.3).sum()} ({(cleaned_df['Rating'] >= 4.3).mean()*100:.1f}%)")


In [ ]:
# KPI 7: Cost Buckets
bins = [0, 200, 300, 400, 500, 1000]
labels = ['≤200', '201-300', '301-400', '401-500', '>500']
cleaned_df['Cost_Bucket'] = pd.cut(cleaned_df['Cost_for_Two'], bins=bins, labels=labels)

cost_bucket_kpi = cleaned_df.groupby('Cost_Bucket').agg(
    Count=('Shop_Name', 'count'),
    Avg_Rating=('Rating', 'mean')
).round(2)
print("=== COST BUCKET ANALYSIS ===")
display(cost_bucket_kpi)


## Quick Visualizations for Insights

In [ ]:
# Area-wise Restaurant Count
plt.figure(figsize=(10, 5))
area_counts = cleaned_df['Area'].value_counts()
sns.barplot(x=area_counts.index, y=area_counts.values, palette='viridis')
plt.title('Number of Restaurants by Area')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Average Rating by Area
plt.figure(figsize=(10, 5))
area_rating = cleaned_df.groupby('Area')['Rating'].mean().sort_values(ascending=False)
sns.barplot(x=area_rating.index, y=area_rating.values, palette='coolwarm')
plt.title('Average Rating by Area')
plt.ylabel('Average Rating')
plt.ylim(3.5, 4.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Top Cuisines
plt.figure(figsize=(12, 6))
sns.barplot(data=top_cuisines.head(10), y='Cuisine', x='Count', palette='mako')
plt.title('Top 10 Most Common Cuisines')
plt.xlabel('Number of Mentions')
plt.tight_layout()
plt.show()


In [ ]:
# Cost vs Rating Scatter
plt.figure(figsize=(10, 6))
sns.scatterplot(data=cleaned_df.dropna(subset=['Rating']), x='Cost_for_Two', y='Rating', hue='Area', alpha=0.7)
plt.title('Cost for Two vs Rating by Area')
plt.xlabel('Cost for Two (₹)')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()


## Export Cleaned Data for Power BI

The cleaned dataset is ready for Power BI. Save it as CSV or Excel.


In [ ]:
# Export cleaned data
output_path = '/workspace/artifacts/swiggy_cleaned.csv'
cleaned_df.to_csv(output_path, index=False)
print(f"Cleaned data exported to: {output_path}")
print(f"Final shape: {cleaned_df.shape}")
print("\nColumns:", cleaned_df.columns.tolist())


In [ ]:
# Recommended KPIs for Power BI Dashboard
print("""
=== RECOMMENDED KPIs FOR POWER BI ===

1. **Overview Cards**:
   - Total Restaurants
   - Average Rating
   - Average Cost for Two
   - % of Restaurants with Rating ≥ 4.0

2. **Area Analysis**:
   - Restaurant count by Area
   - Average Rating by Area
   - Average Cost by Area

3. **Cuisine Insights**:
   - Top Cuisines (frequency)
   - Average Rating by Cuisine
   - Average Cost by Cuisine

4. **Performance Tables**:
   - Top 10 Highest Rated Restaurants
   - Best Value Restaurants (High Rating + Low Cost)
   - Restaurants by Cost Bucket

5. **Filters/Slicers**:
   - Area
   - Rating range
   - Cost range
   - Cuisine (multi-select)

6. **Visuals**:
   - Bar chart: Top cuisines
   - Map or Bar: Area distribution
   - Scatter: Cost vs Rating
   - KPI cards at top
""")
